In [ ]:
!pip install ultralytics
!pip install albumentations
!pip install opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 43.9 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling 

In [ ]:
import os
import zipfile
import shutil
import cv2
import numpy as np
from google.colab import drive
from ultralytics import YOLO
import albumentations as A
from albumentations.pytorch import ToTensorV2
import yaml

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
ZIP_FILE_PATH = '/content/drive/My Drive/Colab Notebooks/labrador_retriever dataset.v1i.yolov11.zip'
EXTRACT_PATH = '/content/dataset_extracted/'
DATASET_ROOT = os.path.join(EXTRACT_PATH, 'dataset')
AUGMENTED_DATA_ROOT = '/content/dataset_augmented/'
AUGMENTED_TRAIN_IMAGES = os.path.join(AUGMENTED_DATA_ROOT, 'train', 'images')
AUGMENTED_TRAIN_LABELS = os.path.join(AUGMENTED_DATA_ROOT, 'train', 'labels')
DATA_YAML_PATH = os.path.join(DATASET_ROOT, 'data.yaml')
INPUT_VIDEO_PATH = '/content/drive/My Drive/Colab Notebooks/your_video.mp4'
OUTPUT_VIDEO_PATH = '/content/drive/My Drive/Colab Notebooks/output_video.mp4'

print("Розархівування (ZIP_FILE_PATH) до (EXTRACT_PATH)...")

if os.path.exists(EXTRACT_PATH):
    print("Папка (EXTRACT_PATH) вже існує. Видаляємо її.")
    shutil.rmtree(EXTRACT_PATH)

os.makedirs(EXTRACT_PATH, exist_ok=True)

with zipfile.ZipFile(ZIP_FILE_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

print("Розпакування завершено.")

extracted_items = os.listdir(EXTRACT_PATH)
print(f"Вміст (EXTRACT_PATH): {extracted_items}")

if len(extracted_items) == 1 and os.path.isdir(os.path.join(EXTRACT_PATH, extracted_items[0])):
    DATASET_ROOT = os.path.join(EXTRACT_PATH, extracted_items[0])
    print(f"Оновлено DATASET_ROOT до: {DATASET_ROOT}")
else:
    DATASET_ROOT = EXTRACT_PATH
    print(f"DATASET_ROOT залишається: {DATASET_ROOT}. Перевірте структуру розпакованих файлів.")

DATA_YAML_PATH = os.path.join(DATASET_ROOT, 'data.yaml')
print("Використовуємо data.yaml за шляхом: {DATA_YAML_PATH}")

Розархівування (ZIP_FILE_PATH) до (EXTRACT_PATH)...
Розпакування завершено.
Вміст (EXTRACT_PATH): ['README.dataset.txt', 'README.roboflow.txt', 'train', 'valid', 'data.yaml', 'test']
DATASET_ROOT залишається: /content/dataset_extracted/. Перевірте структуру розпакованих файлів.
Використовуємо data.yaml за шляхом: {DATA_YAML_PATH}


In [ ]:
print("\nПідготовка до аугментації та копіювання даних...")

if os.path.exists(AUGMENTED_DATA_ROOT):
    print("Папка (AUGMENTED_DATA_ROOT) вже існує. Видаляємо її.")
    shutil.rmtree(AUGMENTED_DATA_ROOT)

os.makedirs(AUGMENTED_TRAIN_IMAGES, exist_ok=True)
os.makedirs(AUGMENTED_TRAIN_LABELS, exist_ok=True)

with open(DATA_YAML_PATH, 'r') as f:
    data_yaml_content = yaml.safe_load(f)

class_names = data_yaml_content['names']
num_classes = data_yaml_content['nc']

print(f"Завантажено інформацію з {DATA_YAML_PATH}:")
print(f"Кількість класів (nc): {num_classes}")
print(f"Назви класів (names): {class_names}")

ORIGINAL_TRAIN_IMAGES = os.path.join(DATASET_ROOT, 'train', 'images')
ORIGINAL_TRAIN_LABELS = os.path.join(DATASET_ROOT, 'train', 'labels')
ORIGINAL_VAL_IMAGES = os.path.join(DATASET_ROOT, 'valid', 'images')
ORIGINAL_VAL_LABELS = os.path.join(DATASET_ROOT, 'valid', 'labels')
ORIGINAL_TEST_IMAGES = os.path.join(DATASET_ROOT, 'test', 'images')
ORIGINAL_TEST_LABELS = os.path.join(DATASET_ROOT, 'test', 'labels')

if not os.path.exists(ORIGINAL_TRAIN_IMAGES) or not os.path.exists(ORIGINAL_TRAIN_LABELS):
    print(
        "Помилка: Не знайдено оригінальні папки train/images ({ORIGINAL_TRAIN_IMAGES}) "
        f"або train/labels ({ORIGINAL_TRAIN_LABELS}) у розпакованому датасеті."
    )
    print("Перевірте структуру вашого zip-архіву та змінну DATASET_ROOT.")
    exit()


Підготовка до аугментації та копіювання даних...
Завантажено інформацію з /content/dataset_extracted/data.yaml:
Кількість класів (nc): 1
Назви класів (names): ['labrador_retriever']


In [ ]:
print("\nКопіювання оригінальних даних та запуск аугментації...")

transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=15, p=0.3),
    A.RGBShift(r_shift_limit=10, g_shift_limit=10, b_shift_limit=10, p=0.2),
    A.Blur(blur_limit=3, p=0.1),
    A.augmentations.geometric.transforms.Affine(scale=(0.8, 1.2), translate_percent=(0.1, 0.1), rotate=(-15, 15), p=0.3),
    A.OpticalDistortion(p=0.1),
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels'], min_visibility=0.3))

num_augmentations_per_image = 3
num_images_to_augment = 5000

image_files = [f for f in os.listdir(ORIGINAL_TRAIN_IMAGES) if
               f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
image_files.sort()

print(f"Знайдено {len(image_files)} оригінальних зображень у тренувальному наборі.")
print(
    f"Аугментація буде проведена для перших {min(len(image_files), num_images_to_augment)} зображень.")

for i, image_name in enumerate(image_files):
    original_img_path = os.path.join(ORIGINAL_TRAIN_IMAGES, image_name)
    label_name = os.path.splitext(image_name)[0] + '.txt'
    original_label_path = os.path.join(ORIGINAL_TRAIN_LABELS, label_name)

    combined_img_path = os.path.join(AUGMENTED_TRAIN_IMAGES, image_name)
    combined_label_path = os.path.join(AUGMENTED_TRAIN_LABELS, label_name)

    shutil.copy(original_img_path, combined_img_path)

    if os.path.exists(original_label_path):
        shutil.copy(original_label_path, combined_label_path)

    if i < num_images_to_augment:
        image = cv2.imread(original_img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        bboxes = []
        class_labels = []

        if os.path.exists(original_label_path):
            with open(original_label_path, 'r') as f:
                for line in f.readlines():
                    parts = line.strip().split()
                    if len(parts) == 5:
                        class_id = int(parts[0])
                        bbox = [float(p) for p in parts[1:]]
                        bboxes.append(bbox)
                        class_labels.append(class_id)

        for j in range(num_augmentations_per_image):
            try:
                augmented = transform(image=image, bboxes=bboxes, class_labels=class_labels)
                aug_img = augmented['image']
                aug_bboxes = augmented['bboxes']
                aug_class_labels = augmented['class_labels']

                # Якщо після аугментації рамки зникли — пропускаємо збереження
                if len(aug_bboxes) == 0:
                    print(f"Аугментація {base_name}_aug_{j} пропущена: усі рамки були втрачені або обрізані.")
                    continue

                base_name = os.path.splitext(image_name)[0]
                aug_img_name = f"{base_name}_aug_{j}.png"
                aug_label_name = f"{base_name}_aug_{j}.txt"

                aug_img_path = os.path.join(AUGMENTED_TRAIN_IMAGES, aug_img_name)
                aug_label_path = os.path.join(AUGMENTED_TRAIN_LABELS, aug_label_name)

                cv2.imwrite(aug_img_path, cv2.cvtColor(aug_img, cv2.COLOR_RGB2BGR))

                with open(aug_label_path, 'w') as f:
                    for k, bbox in enumerate(aug_bboxes):
                        if len(bbox) == 4 and aug_class_labels[k] is not None:
                            class_id = aug_class_labels[k]
                            f.write(f"{class_id} {bbox[0]:.6f} {bbox[1]:.6f} {bbox[2]:.6f} {bbox[3]:.6f}\n")
                        else:
                            print(f"Попередження: Відкинуто некоректну рамку після аугментації в файлі {aug_label_name}")

            except Exception as e:
                print(f"Аугментація {image_name} (ітерація {j}) пропущена через помилку: {str(e)}")
                continue

    if (i + 1) % 100 == 0:
        print(
            f"Опрацьовано {i + 1}/{len(image_files)} оригінальних зображень")

print("\nКопіювання оригінальних даних та аугментація завершені.")
print(f"Всі тренувальні дані (оригінали + аугментовані) збережено у {AUGMENTED_TRAIN_IMAGES}")
print(
    f"Кількість файлів зображень у новому тренувальному наборі: {len(os.listdir(AUGMENTED_TRAIN_IMAGES))}")
print(
    f"Кількість файлів міток у новому тренувальному наборі: {len(os.listdir(AUGMENTED_TRAIN_LABELS))}")


Копіювання оригінальних даних та запуск аугментації...
Знайдено 221 оригінальних зображень у тренувальному наборі.
Аугментація буде проведена для перших 221 зображень.


/usr/local/lib/python3.11/dist-packages/albumentations/core/validation.py:111: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


Опрацьовано 100/221 оригінальних зображень
Опрацьовано 200/221 оригінальних зображень

Копіювання оригінальних даних та аугментація завершені.
Всі тренувальні дані (оригінали + аугментовані) збережено у /content/dataset_augmented/train/images
Кількість файлів зображень у новому тренувальному наборі: 884
Кількість файлів міток у новому тренувальному наборі: 884


In [ ]:
print("\nОновлення data.yaml...")

original_data_yaml_path = os.path.join(DATASET_ROOT, 'data.yaml')

with open(original_data_yaml_path, 'r') as f:
    data_yaml_content = yaml.safe_load(f)

data_yaml_content['train'] = os.path.relpath(AUGMENTED_TRAIN_IMAGES,
                                            os.path.dirname(DATA_YAML_PATH))

if os.path.exists(ORIGINAL_VAL_IMAGES):
    data_yaml_content['valid'] = os.path.relpath(ORIGINAL_VAL_IMAGES,
                                                os.path.dirname(DATA_YAML_PATH))
else:
    print("Попередження: Не знайдено оригінальну папку валідації "
          f"({ORIGINAL_VAL_IMAGES}).")
    data_yaml_content['valid'] = 'path/to/your/original/valid/images'

if os.path.exists(ORIGINAL_TEST_IMAGES):
    data_yaml_content['test'] = os.path.relpath(ORIGINAL_TEST_IMAGES,
                                              os.path.dirname(DATA_YAML_PATH))
else:
    print("Попередження: Не знайдено оригінальну папку тестування "
          f"({ORIGINAL_TEST_IMAGES}).")
    data_yaml_content['test'] = 'path/to/your/original/test/images'

with open(DATA_YAML_PATH, 'w') as f:
    yaml.dump(data_yaml_content, f)

print(f"Оновлений data.yaml збережено за шляхом: {DATA_YAML_PATH}")
print("Зміст оновленого data.yaml:")
print(yaml.dump(data_yaml_content))


Оновлення data.yaml...
Оновлений data.yaml збережено за шляхом: /content/dataset_extracted/data.yaml
Зміст оновленого data.yaml:
names:
- labrador_retriever
nc: 1
roboflow:
  license: CC BY 4.0
  project: labrador_retriever-dataset
  url: https://universe.roboflow.com/firstworkspace-tmxgd/labrador_retriever-dataset/dataset/2
  version: 2
  workspace: firstworkspace-tmxgd
test: test/images
train: ../dataset_augmented/train/images
val: ../valid/images
valid: valid/images



In [ ]:
print("\nЗапуск навчання YOLOV11...")

model = YOLO('yolov11n.pt')

EPOCHS = 10
IMG_SIZE = 640
BATCH_SIZE = 80
PROJECT_NAME = 'labrador_detection'
EXPERIMENT_NAME = 'yolov11_subset_augmented'

results = model.train(
    data=DATA_YAML_PATH,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    project=PROJECT_NAME,
    name=EXPERIMENT_NAME
)

print("\nНавчання YOLOv11 завершено.")

BEST_MODEL_PATH = os.path.join('/content/runs/detect', PROJECT_NAME,
                              EXPERIMENT_NAME, 'weights', 'best.pt')
print(f"Краща модель збережена за шляхом: {BEST_MODEL_PATH}")


Запуск навчання YOLOV8...


100%|██████████| 6.25M/6.25M [00:00<00:00, 444MB/s]


Ultralytics 8.3.128 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=80, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset_extracted/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov11_subset_augmented, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=

100%|██████████| 755k/755k [00:00<00:00, 164MB/s]

Overriding model.yaml nc=80 with nc=1

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralytics

...

      10/10      9.33G     0.4762     0.5686     0.9276          7        640: 100%|██████████| 12/12 [00:13<00:00,  1.14s/it]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:00<00:00,  1.30it/s]

                   all         63         94      0.977      0.947      0.973       0.89



10 epochs completed in 0.051 hours.
Optimizer stripped from labrador_detection/yolov11_subset_augmented/weights/last.pt, 6.2MB
Optimizer stripped from labrador_detection/yolov11_subset_augmented/weights/best.pt, 6.2MB

Validating labrador_detection/yolov11_subset_augmented/weights/best.pt...
Ultralytics 8.3.128 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:00<00:00,  1.15it/s]


                   all         63         94      0.977      0.947      0.973      0.889
Speed: 0.2ms preprocess, 2.2ms inference, 0.0ms loss, 3.0ms postprocess per image
Results saved to labrador_detection/yolov11_subset_augmented

Навчання YOLOv8 завершено.
Краща модель збережена за шляхом: /content/runs/detect/labrador_detection/yolov11_subset_augmented/weights/best.pt


In [ ]:
print("\nЗапуск обробки відео...")
if not os.path.exists('/content/labrador_detection/yolov11_subset_augmented/weights/best.pt'):
    print(f"Помилка: Не знайдено навчену модель за шляхом /content/labrador_detection/yolov11_subset_augmented/weights/best.pt.")
    print("Перевірте, чи навчання пройшло успішно і шлях до моделі вірний.")
else:
    try:
        trained_model = YOLO('/content/labrador_detection/yolov11_subset_augmented/weights/best.pt')
        print(f"Навчена модель завантажена з /content/labrador_detection/yolov11_subset_augmented/weights/best.pt.")
        cap = cv2.VideoCapture(INPUT_VIDEO_PATH)
        if not cap.isOpened():
            print(f"Помилка: Не вдалося відкрити відеофайл за шляхом {INPUT_VIDEO_PATH}.")
        else:
            frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
            fps = int(cap.get(cv2.CAP_PROP_FPS))
            fourcc = cv2.VideoWriter_fourcc(*'mp4v')
            out = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, fps, (frame_width, frame_height))
            if not out.isOpened():
                print(f"Помилка: Не вдалося створити об'єкт для запису відео за шляхом {OUTPUT_VIDEO_PATH}.")
                print("Перевірте шлях та доступність місця на Google Drive.")
                cap.release()
            else:
                print(f"Розпочато обробку відео. Зберігаємо результат у {OUTPUT_VIDEO_PATH}")
                frame_count = 0
                while cap.isOpened():
                    ret, frame = cap.read()
                    if not ret:
                        break
                    frame_count += 1
                    if frame_count % 100 == 0:
                        print(f"Опрацьовано {frame_count} кадрів...")
                    results = trained_model(frame, imgsz=IMG_SIZE, conf=0.5)
                    annotated_frame = results[0].plot()
                    out.write(annotated_frame)
                cap.release()
                out.release()
                print("\nОбробка відео завершена.")
    except Exception as e:
        print(f"Сталася помилка під час обробки відео: {e}")


Запуск обробки відео...
Навчена модель завантажена з /content/labrador_detection/yolov11_subset_augmented/weights/best.pt.
Розпочато обробку відео. Зберігаємо результат у /content/drive/My Drive/Colab Notebooks/output_video.mp4

0: 640x384 1 labrador_retriever, 9.5ms
Speed: 2.7ms preprocess, 9.5ms inference, 1.6ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 labrador_retriever, 7.4ms
Speed: 3.2ms preprocess, 7.4ms inference, 1.5ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 labrador_retriever, 8.1ms
Speed: 3.0ms preprocess, 8.1ms inference, 1.5ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 labrador_retriever, 9.6ms
Speed: 3.1ms preprocess, 9.6ms inference, 1.5ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 labrador_retriever, 6.7ms
Speed: 3.3ms preprocess, 6.7ms inference, 1.5ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 1 labrador_retriever, 9.8ms
Speed: 3.0ms preprocess, 9.8ms inference, 1.4ms pos